In [2]:
import cv2
import numpy as np

In [31]:
import cv2
import numpy as np
from collections import deque

TRACK_LENGTH = 10
VX_TOLERANCE = 1.5
AY_TOLERANCE = 1.5

cap = cv2.VideoCapture("video1.mp4")

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

positions = deque(maxlen=TRACK_LENGTH)

def is_projectile_motion(points):
    if len(points) < TRACK_LENGTH:
        return False

    points = np.array(points)

    vx = np.diff(points[:, 0])
    vy = np.diff(points[:, 1])
    ay = np.diff(vy)

    vx_const = np.std(vx) < VX_TOLERANCE
    ay_const = np.std(ay) < AY_TOLERANCE

    return vx_const and ay_const


while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, gray,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    mask = mag > 2.0

    if np.any(mask):
        ys, xs = np.where(mask)

        cx = int(xs.mean())
        cy = int(ys.mean())

        positions.append((cx, cy))

        cv2.circle(frame, (cx, cy), 6, (0, 255, 255), -1)

    if is_projectile_motion(positions):
        cv2.putText(
            frame,
            "ELHAJITOTT TEST",
            (30, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2,
        )

        for i in range(1, len(positions)):
            cv2.line(
                frame,
                positions[i - 1],
                positions[i],
                (0, 0, 255),
                2
            )

    cv2.imshow("Optical Flow Projectile Detection", frame)

    prev_gray = gray

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

w, h = 640, 480
fps = 30
duration = 20
frames = fps * duration

out = cv2.VideoWriter(
    "synthetic_throw.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

# kezdeti feltételek
x0, y0 = 50, 180
vx = 8
vy = -15
g = 1

x, y = x0, y0

for t in range(frames):
    frame = np.zeros((h, w, 3), dtype=np.uint8)

    x = int(x0 + vx * t)
    y = int(y0 + vy * t + 0.5 * g * t * t)

    if 0 < x < w and 0 < y < h:
        cv2.circle(frame, (x, y), 10, (255, 255, 255), -1)

    out.write(frame)

out.release()
print("synthetic_throw.mp4 kész")


synthetic_throw.mp4 kész
